# Medical-Insurance-Cost-Prediction

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.svm import SVR

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Load Data

In [61]:
df = pd.read_csv("data/insurance.csv")

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## EDA

In [62]:
df.shape

(1338, 7)

In [63]:
print(df.columns.tolist())

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']


In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [65]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


Check missing values

In [66]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Check duplicates

In [67]:
print(df.duplicated().sum())

1


In [68]:
for column in df.columns:
    print(f"{column}: {df[column].nunique()} unique values")

age: 47 unique values
sex: 2 unique values
bmi: 548 unique values
children: 6 unique values
smoker: 2 unique values
region: 4 unique values
charges: 1337 unique values


In [69]:
print("\nSex:")
print(df["sex"].value_counts())

print("\nSmoker:")
print(df["smoker"].value_counts())

print("\nRegion:")
print(df["region"].value_counts())


Sex:
sex
male      676
female    662
Name: count, dtype: int64

Smoker:
smoker
no     1064
yes     274
Name: count, dtype: int64

Region:
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


## Data Preprocessing

Remove duplicates

In [70]:
df = df.drop_duplicates()

print(df.duplicated().sum())

0


Separate features and target

In [71]:
X = df.drop("charges", axis=1)

y = df["charges"]

In [72]:
print("Features:")
print(X.head())

Features:
   age     sex     bmi  children smoker     region
0   19  female  27.900         0    yes  southwest
1   18    male  33.770         1     no  southeast
2   28    male  33.000         3     no  southeast
3   33    male  22.705         0     no  northwest
4   32    male  28.880         0     no  northwest


In [73]:
print("Target:")
print(y.head())

Target:
0    16884.92400
1     1725.55230
2     4449.46200
3    21984.47061
4     3866.85520
Name: charges, dtype: float64


In [74]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

Numerical Columns: Index(['age', 'bmi', 'children'], dtype='str')
Categorical Columns: Index(['sex', 'smoker', 'region'], dtype='str')


C:\Users\13ver\AppData\Local\Temp\ipykernel_11636\2452935477.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns


Train/test split

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nData split into training and testing sets")
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)


Data split into training and testing sets
Training size: (1069, 6)
Testing size: (268, 6)


Create the pipeline

In [76]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


## Train

In [84]:
models = {

    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(alpha=1.0),

    "Lasso Regression": Lasso(alpha=0.01),

    "Elastic Net": ElasticNet(
        alpha=0.01,
        l1_ratio=0.5
    ),

    "SVR": SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1
    )
}

In [95]:
results = []

trained_models = {}

for model_name, model in models.items():

    print()
    print("=" * 60)
    print(f"TRAINING: {model_name}")
    print("=" * 60)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)
    
    predictions = pipeline.predict(X_test)
    

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    print()
    print(f"Model : {model_name}")
    print(f"MAE   : {mae:.2f}")
    print(f"MSE   : {mse:.2f}")
    print(f"RMSE  : {rmse:.2f}")
    print(f"R²    : {r2:.4f}")

    results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[model_name] = pipeline

    print("Model training completed!")


TRAINING: Linear Regression

Model : Linear Regression
MAE   : 4177.05
MSE   : 35478020.68
RMSE  : 5956.34
R²    : 0.8069
Model training completed!

TRAINING: Ridge Regression

Model : Ridge Regression
MAE   : 4185.41
MSE   : 35572594.48
RMSE  : 5964.28
R²    : 0.8064
Model training completed!

TRAINING: Lasso Regression

Model : Lasso Regression
MAE   : 4177.05
MSE   : 35478162.97
RMSE  : 5956.35
R²    : 0.8069
Model training completed!

TRAINING: Elastic Net

Model : Elastic Net
MAE   : 4221.25
MSE   : 35996967.56
RMSE  : 5999.75
R²    : 0.8041
Model training completed!

TRAINING: SVR

Model : SVR
MAE   : 6879.90
MSE   : 172623725.00
RMSE  : 13138.63
R²    : 0.0606
Model training completed!


## Compare All Models

In [93]:
results_df = pd.DataFrame(results)

comparison = results_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

print(comparison)

               Model          MAE           MSE          RMSE        R2
0  Linear Regression  4177.045561  3.547802e+07   5956.342894  0.806929
1   Lasso Regression  4177.053711  3.547816e+07   5956.354839  0.806928
2   Ridge Regression  4185.406967  3.557259e+07   5964.276526  0.806414
3        Elastic Net  4221.250989  3.599697e+07   5999.747291  0.804105
4                SVR  6879.902245  1.726237e+08  13138.634823  0.060582


Best Model

In [92]:
best_model_name = comparison.iloc[0]["Model"]

best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)

print(
    "RMSE:",
    comparison.iloc[0]["RMSE"]
)

print(
    "R²:",
    comparison.iloc[0]["R2"]
)

Best Model: Linear Regression
RMSE: 5956.342894363587
R²: 0.8069287081198012


Save Model

In [85]:
model_filename = "medical_cost_model.pkl"

joblib.dump(
    best_model,
    model_filename
)

print()
print("Best model saved successfully!")
print("File:", model_filename)


Best model saved successfully!
File: medical_cost_model.pkl


In [ ]:
feature_names = (
    lasso_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print(feature_names)

## Predict

In [87]:
loaded_model = joblib.load(
    "medical_cost_model.pkl"
)

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [88]:
new_patient = pd.DataFrame({
    "age": [35],
    "sex": ["male"],
    "bmi": [28.5],
    "children": [2],
    "smoker": ["no"],
    "region": ["southeast"]
})

In [90]:
prediction = loaded_model.predict(
    new_patient
)


print(f"Predicted Medical Cost: ${prediction[0]:.2f}")

Predicted Medical Cost: $6803.27
